# 04 — Screening scenarios

Comparison of alternative active case finding algorithms, each delivered as a two-year campaign
starting in 2026, against a no-screening comparator:

| Scenario | Algorithm |
| --- | --- |
| `ssx` | Symptom screening only |
| `cxr` | Universal chest X-ray, treatment on radiological diagnosis |
| `cxr_xpert` | CXR triage with Xpert confirmation |
| `cxr_xpert_tpt` | CXR + Xpert, plus TST-guided preventive therapy |
| `cxr_xpert_tpt_cov40` / `_cov90` | The full algorithm at 40% and 90% coverage |

Because the model tracks the number of tests performed by type, the scenarios can be compared on
both **impact** (TB episodes and deaths averted) and **resource use** (tests consumed). As in
notebook 03, every scenario is run once on the single maximum likelihood parameter set saved by
notebook 02, so the comparison is deterministic.

Contents:

1. **Algorithms and scenarios** — what is being compared.
2. **Run** — load the MLE parameters and run every scenario once.
3. **Epidemiological impact** — burden trajectories and case detection.
4. **Resource use and efficiency** — tests performed, and tests per TB episode averted.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import tbtoy.plotting as pl
import tbtoy.runner_tools as rt
from tbtoy.config import DEFAULT_MODEL_CONFIG
from tbtoy.interventions import SCREENING_TOOLS
from tbtoy.paths import MLE_PARAMS_PATH
from tbtoy.scenarios import SCREENING_SCENARIOS, get_scenario_names

sc_names = get_scenario_names(SCREENING_SCENARIOS)
end_year = DEFAULT_MODEL_CONFIG["end_time"]


## 1. Screening algorithms and scenarios

A screening **tool** defines which compartments it can detect (and with what sensitivity), where
screen-positive individuals are moved to, and how many tests of each type are consumed per person
screened. A screening **program** attaches a tool to a time window and a total coverage; a scenario
can combine several concurrent programs (e.g. disease screening plus preventive therapy).


In [ ]:
pd.DataFrame(
    [
        {
            "detects": ", ".join(tool.sensitivities),
            "moves to": tool.dest_comp,
            "tests used": ", ".join(tool.tests_per_screen),
        }
        for tool in SCREENING_TOOLS.values()
    ],
    index=pd.Index(SCREENING_TOOLS, name="tool"),
)


In [ ]:
rows = []
for scenario in SCREENING_SCENARIOS:
    programs = scenario.screening_programs
    first = programs[0] if programs else None
    rows.append(
        {
            "scenario": scenario.sc_name,
            "tools": ", ".join(prog.tool.name for prog in programs) or "none",
            "window": f"{first.start_time:.0f}-{first.end_time:.0f}" if first else "-",
            "coverage (%)": first.coverage_perc if first else 0.0,
            "description": scenario.description,
        }
    )

pd.DataFrame(rows, index=[scenario.sc_id for scenario in SCREENING_SCENARIOS])


## 2. Run the scenarios

As in notebook 03, every scenario is run once with the maximum likelihood parameter set — reused
from `outputs/calibration/mle_params.yml` if notebook 02 has been run, and otherwise re-optimised
here on the baseline scenario and saved.


In [ ]:
bcm_dict = rt.build_bcm_dict(SCREENING_SCENARIOS)

print("Reusing MLE from notebook 02" if MLE_PARAMS_PATH.exists() else "No saved MLE found, optimising here")
mle_params = rt.get_mle_params(bcm_dict["baseline"], budget=1000)

scenario_outputs = rt.run_scenarios_single_params(bcm_dict, mle_params)

output_dir = rt.create_output_dir("screening_scenarios")
for sc_id, derived_outputs in scenario_outputs.items():
    derived_outputs.to_parquet(output_dir / f"derived_outputs_{sc_id}.parquet")

pd.Series(mle_params, name="MLE").to_frame()


In [ ]:
# screening activity is confined to the campaign window; check it lands where expected
campaign_years = slice(2025, 2029)
pd.DataFrame(
    {
        sc_names[sc_id]: derived_outputs.loc[campaign_years, "n_screening_encounters"].round(0)
        for sc_id, derived_outputs in scenario_outputs.items()
    }
).rename_axis("year")


## 3. Epidemiological impact

A campaign produces a sharp, temporary increase in treatment starts, followed by a drop in
prevalence and, with a lag, in incidence. Whether that gain is sustained after the campaign ends
depends on how much transmission the campaign interrupts.


In [ ]:
burden_outputs = ["tb_incidence_per100k", "tb_prevalence_per100k", "tb_mortality_per100k", "viable_tbi_prevalence_perc"]

pl.plot_scenarios_single_params(scenario_outputs, burden_outputs, sc_names, x_lim=(2020, end_year));


In [ ]:
cascade_outputs = ["screening_tb_detections", "tb_treatment_starts", "tpt_completions", "perc_prev_subclinical"]

pl.plot_scenarios_single_params(scenario_outputs, cascade_outputs, sc_names, x_lim=(2020, end_year));


## 4. Resource use and efficiency

Algorithms differ far more in what they consume than in what they achieve, so impact is reported
alongside the number of tests performed, and combined into a simple efficiency measure: tests per
TB episode averted (lower is better).


In [ ]:
diff_df = rt.calculate_diff_outputs_single_params(scenario_outputs, ref_sc="baseline", end_year=end_year)
diff_df.to_parquet(output_dir / "diff_outputs_df.parquet")

fig, axes = plt.subplots(1, 2, figsize=(13, 0.6 * len(diff_df) + 1.8))
for ax, output in zip(axes, ["TB_averted", "deaths_averted"]):
    pl.plot_diff_outputs_single_params(diff_df, output, sc_names, ax=ax)
fig.suptitle(f"Cumulative impact by {int(end_year)}, relative to no screening", fontsize=11)
fig.tight_layout()


In [ ]:
resource_outputs = ["cum_n_screening_encounters", "cum_n_tests_symptom_screen", "cum_n_tests_cxr", "cum_n_tests_xpert", "cum_n_tests_tst"]
cumulative_df = rt.calculate_cumulative_outputs_single_params(
    scenario_outputs, outputs=resource_outputs, end_year=end_year
)
cumulative_df.to_parquet(output_dir / "cumulative_outputs_df.parquet")

cumulative_df.rename(index=sc_names).round(0)


In [ ]:
pl.plot_cumulative_outputs_single_params(cumulative_df, resource_outputs, sc_names, n_col=3);


In [ ]:
baseline_latest = scenario_outputs["baseline"].loc[end_year]

efficiency = {}
for sc_id, derived_outputs in scenario_outputs.items():
    if sc_id == "baseline":
        continue
    latest = derived_outputs.loc[end_year]
    tb_averted = baseline_latest["cum_tb_incidence"] - latest["cum_tb_incidence"]
    deaths_averted = baseline_latest["cum_tb_mortality"] - latest["cum_tb_mortality"]
    efficiency[sc_names[sc_id]] = {
        "TB averted": tb_averted,
        "Deaths averted": deaths_averted,
        "Tests performed": latest["cum_n_tests"],
        "Tests per TB episode averted": latest["cum_n_tests"] / tb_averted,
        "Tests per death averted": latest["cum_n_tests"] / deaths_averted,
    }

pd.DataFrame(efficiency).T.round(1)


### Coverage sensitivity

The last three scenarios deliver the same algorithm (CXR + Xpert + TST/TPT) at 40%, 70% and 90%
coverage, which shows how impact and resource use scale with the reach of the campaign.

> Caveats: with homogeneous mixing and no age structure, screening reaches everyone with equal
> probability and preventive therapy benefits are approximated — both of which tend to flatter
> active case finding. All input data here are dummy values.


In [ ]:
coverage_scenarios = {"cxr_xpert_tpt_cov40": 40.0, "cxr_xpert_tpt": 70.0, "cxr_xpert_tpt_cov90": 90.0}

coverages = list(coverage_scenarios.values())
tb_averted = [diff_df.loc[sc_id, "TB_averted"] for sc_id in coverage_scenarios]
tests = [scenario_outputs[sc_id].loc[end_year, "cum_n_tests"] for sc_id in coverage_scenarios]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(coverages, tb_averted, marker="o", color="tab:blue")
axes[0].set_ylabel(pl.get_title("TB_averted"))
axes[1].plot(coverages, [t / a for t, a in zip(tests, tb_averted)], marker="o", color="tab:red")
axes[1].set_ylabel("Tests per TB episode averted")

for ax in axes:
    ax.set_xlabel("Campaign coverage (%)")
    ax.set_xticks(coverages)
    ax.grid(alpha=0.3)
    ax.set_ylim(bottom=0.0)
fig.suptitle("CXR + Xpert + TST/TPT, by coverage", fontsize=11)
fig.tight_layout()
